# Introduction

This noteebook contains my attempt in creating a software able to forecast the winning team of a football game with machine learning models, using the odds of bookmakers and several performance statistics of the teams involved. 

This notebook has the porpouse to illustrate the complete workflow of this project, with detailed explenation of each chunk of code and of each decision made. 

The final software is created with specific Python scripts and it is thought to find valuable bets based on the predictions. 

In the description of this GitHub repository an explenation of the software usage is provided

## 1. The Data

Since this is a Machine Learning problem the most important thing is gathering data that can solve the problem we are trying to solve. 
In this case it was decided to use data of each game from the 21-22 season to the 25-26 one of the 5 top European leagues, along with the the odds of every possible final result. 

This first chunck doesn't need to be run since it contains the code used to download all the training data, that I shared in the 'data' directory.

In [ ]:
import soccerdata as sd
import pandas as pd
import time 
from pathlib import Path


## let's start by getting all the games played in the season from 21-22 season
## in the european top 5 leagues and the key features of each game

leagues = ['ESP-La Liga', 'ITA-Serie A', 'ENG-Premier League', 'GER-Bundesliga', 'FRA-Ligue 1']
seasons = ['2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']

games_final = []

#collect all the data about games 
for league in leagues:
    games = sd.Understat(league, seasons)
    games_final.append(games.read_schedule())

    print('{} succesfully saved'.format(league))
    time.sleep(15)


games_df = pd.concat(games_final, axis=0)
games_df = games_df.reset_index()


games_df.to_csv('data/games.csv', index=False, sep=';')



#### And now let's import all the odds
'''
odds_final = []

for league in leagues: 
    for season in seasons:
        odds = sd.MatchHistory(leagues=league, seasons=season)

        odds_final.append(odds.read_games())
        print('{} {} succesfully saved'.format(league, season))
        time.sleep(5)


odds_df = pd.concat(odds_final, axis=0)
odds_df = odds_df.reset_index()

odds_df.to_csv('data/odds.csv', index=False, sep=';')

'''

### The previous code is the fastest way possible to collect data, but it may not work, 
## try manually downloading data from https://www.football-data.co.uk/ then execute the following code:

directory = Path('C:/Users/emanu/OneDrive/Desktop/progetti/value_betting_software/data/tmp') # write your own path

df_list = []

for file in directory.glob('*.csv'):
    df = pd.read_csv(file)
    df_list.append(df)

odds_df = pd.concat(df_list, axis=0, ignore_index=True)
odds_df.to_csv('data/odds.csv', index=False)


[09/18/26 12:28:11] INFO     Saving cached data to C:\Users\emanu\soccerdata\data\Understat          ]8;id=314351;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=930578;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py#249\249]8;;\

ESP-La Liga succesfully saved


[09/18/26 12:28:26] INFO     Saving cached data to C:\Users\emanu\soccerdata\data\Understat          ]8;id=905235;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=535341;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py#249\249]8;;\

ITA-Serie A succesfully saved


[09/18/26 12:28:42] INFO     Saving cached data to C:\Users\emanu\soccerdata\data\Understat          ]8;id=687675;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=435135;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py#249\249]8;;\

ENG-Premier League succesfully saved


[09/18/26 12:28:58] INFO     Saving cached data to C:\Users\emanu\soccerdata\data\Understat          ]8;id=787340;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=749151;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py#249\249]8;;\

GER-Bundesliga succesfully saved


[09/18/26 12:29:14] INFO     Saving cached data to C:\Users\emanu\soccerdata\data\Understat          ]8;id=852143;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=835696;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py#249\249]8;;\

FRA-Ligue 1 succesfully saved


We have several different dataframe, so it is time to merge them.

In [18]:
import pandas as pd
import numpy as np

### In this section the data is merged in a single dataset and some preprocessing will be performed

games = pd.read_csv('data/games.csv', sep = ';')
odds = pd.read_csv('data/odds.csv')


## First let's select only the features that are needed

odds_filtered = odds.loc[:, ['Date', 'HomeTeam','AwayTeam', 'B365H', 'B365D', 'B365A', 'HS', 'AS', 'HST', 'AST', 'FTR', 'HC', 'AC']] 

games_filtered = games.loc[:, ['season', 'date', 'game', 'home_team','away_team','home_goals', 'away_goals', 'home_xg', 'away_xg']]


## Since the teams' names are not the same in the two df, let's address this problem 

games_teams = sorted(games_filtered['home_team'].unique())

odds_teams = sorted(odds_filtered['HomeTeam'].unique())

### The following dictionary contains all the different names, so we can replace them in the dataframe
different_names_dict = {
    'AC Milan': 'Milan',
    'Arminia Bielefeld': 'Bielefeld',
    'Athletic Club': 'Ath Bilbao',
    'Atletico Madrid': 'Ath Madrid',
    'Bayer Leverkusen': 'Leverkusen',
    'Borussia Dortmund': 'Dortmund',
    "Borussia M.Gladbach": "M'gladbach",
    'Celta Vigo': 'Celta',
    'Clermont Foot': 'Clermont',
    'Eintracht Frankfurt': 'Ein Frankfurt',
    'Espanyol': 'Espanol',
    'FC Cologne': 'FC Koln',
    'FC Heidenheim': 'Heidenheim',
    'Greuther Fuerth': 'Greuther Furth',
    'Hamburger SV': 'Hamburg',
    'Hertha Berlin': 'Hertha',
    'Mainz 05': 'Mainz',
    'Manchester City': 'Man City',
    'Manchester United': 'Man United',
    'Newcastle United': 'Newcastle',
    "Nottingham Forest": "Nott'm Forest",
    'Paris Saint Germain': 'Paris SG',
    'Parma Calcio 1913': 'Parma',
    'RasenBallsport Leipzig': 'RB Leipzig',
    'Rayo Vallecano': 'Vallecano',
    'Real Betis': 'Betis',
    'Real Oviedo': 'Oviedo',
    'Real Sociedad': 'Sociedad',
    'Real Valladolid': 'Valladolid',
    'Saint-Etienne': 'St Etienne',
    'St. Pauli': 'St Pauli',
    'VfB Stuttgart': 'Stuttgart',
    'Wolverhampton Wanderers': 'Wolves'
    }


games_filtered['home_team'] = games_filtered['home_team'].replace(different_names_dict)
games_filtered['away_team'] = games_filtered['away_team'].replace(different_names_dict)



## Let's aslso address the problem of the dates that are different in the 2 df
games_filtered['date'] = pd.to_datetime(games_filtered['date']).dt.normalize()

odds_filtered['Date'] = pd.to_datetime(odds_filtered['Date'], dayfirst=True).dt.normalize()


## And now finally merge the data in a single df
final_df = pd.merge(games_filtered, odds_filtered, left_on=['date', 'home_team','away_team'], right_on=['Date', 'HomeTeam','AwayTeam'])


### Now that the data has been cleared it's time for some feature engeneering to obtain all the final feature we want to include

[09/18/26 12:52:08] WARNING  C:\Users\emanu\AppData\Local\Temp\ipykernel_21276\3111765022.py:7:     ]8;id=316677;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py\warnings.py]8;;\:]8;id=982196;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py#110\110]8;;\
                             DtypeWarning: Columns (161) have mixed types. Specify dtype option on                 
                             import or set low_memory=False.                                                       
                               odds = pd.read_csv('data/odds.csv')                                                 
                                                                                                                   

In [19]:
final_df.head()

,season,date,game,home_team,away_team,home_goals,away_goals,home_xg,away_xg,Date,...,B365H,B365D,B365A,HS,AS,HST,AST,FTR,HC,AC
0,2122,2021-08-13,2021-08-13 Valencia-Getafe,Valencia,Getafe,1.0,0.0,1.578610,1.193260,2021-08-13,...,2.55,3.00,3.10,4.0,22.0,2.0,4.0,H,1.0,9.0
1,2122,2021-08-14,2021-08-14 Alaves-Real Madrid,Alaves,Real Madrid,1.0,4.0,1.410970,2.155510,2021-08-14,...,7.00,4.75,1.44,11.0,19.0,4.0,7.0,A,0.0,4.0
2,2122,2021-08-14,2021-08-14 Cadiz-Levante,Cadiz,Levante,1.0,1.0,0.993589,0.915954,2021-08-14,...,2.80,3.25,2.60,7.0,12.0,2.0,3.0,D,2.0,4.0
3,2122,2021-08-14,2021-08-14 Mallorca-Real Betis,Mallorca,Betis,1.0,1.0,0.569578,0.814085,2021-08-14,...,3.30,3.40,2.20,6.0,10.0,2.0,1.0,D,4.0,3.0
4,2122,2021-08-14,2021-08-14 Osasuna-Espanyol,Osasuna,Espanol,0.0,0.0,0.579404,0.583698,2021-08-14,...,2.25,3.20,3.40,14.0,10.0,1.0,3.0,D,4.0,6.0


And now that we have a final dataframe containing everything, let's perform some feature engineering to obtain all the features we need.

In [20]:
## First of all let's get the probability of every outcome (1/odd)

stakes = ['B365H', 'B365D', 'B365A']

for stake in stakes:
    final_df[stake] = 1/final_df[stake]

prov_sum = final_df.B365A + final_df.B365D + final_df.B365H

for stake in stakes:
    final_df[stake] = final_df[stake]/prov_sum


## Now we need to double every game so that we can work properly 
df_home = final_df[['season', 'date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 
                    'away_xg', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC']].copy()


df_home.columns = ['season', 'date', 'team', 'opponent', 'goals_for', 'goals_against', 'xG_for', 'xG_against', 
                   'shots_for', 'shots_against', 'shots_target_for', 'shots_target_against', 'corners_for', 'corners_against']

df_home['is_home'] = 1


df_away = final_df[['season', 'date', 'away_team', 'home_team', 'away_goals', 'home_goals', 
                    'away_xg', 'home_xg', 'AS', 'HS', 'AST', 'HST', 'AC', 'HC']].copy()

df_away.columns = ['season', 'date', 'team', 'opponent', 'goals_for', 'goals_against', 'xG_for', 'xG_against', 
                   'shots_for', 'shots_against', 'shots_target_for', 'shots_target_against', 'corners_for', 'corners_against']


df_away['is_home'] = 0



df_long = pd.concat([df_home, df_away], axis=0)


## And now lets's order the dataframe by team and date and compute all the feature that could be usefull:
## Rest days since lst match, total goals scored and conceded, total xG of the team and conceded, total points in the last 5 matches and
## points per game

df_long = df_long.sort_values(by=['team', 'date']).reset_index(drop=True)


mask = [df_long.goals_for > df_long.goals_against, 
        df_long.goals_for == df_long.goals_against,
        df_long.goals_for < df_long.goals_against]

df_long['points gained'] = np.select(mask, [3, 1, 0])

groups = df_long.groupby(['season', 'team'])

df_long['rest_days'] = groups['date'].diff().dt.days

df_long['total_goals'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))

df_long['total_xg'] = groups['xG_for'].transform(lambda x: x.cumsum().shift(1))

df_long['total_goals_against'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))

df_long['total_xg_against'] = groups['xG_against'].transform(lambda x: x.cumsum().shift(1))

df_long['last_5'] = groups['points gained'].transform(lambda x: x.shift(1).rolling(5).sum())

df_long['match_played'] = groups.cumcount()

df_long['PPG'] = groups['points gained'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_shots_for'] = groups['shots_for'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_shots_target_for'] = groups['shots_target_for'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_shots_against'] = groups['shots_against'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_shots_target_against'] = groups['shots_target_against'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_corners_for'] = groups['corners_for'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_corners_against'] = groups['corners_against'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']


############ Now it's time to merge this new features with the new ones

df = pd.merge(final_df, df_long, left_on=['date', 'home_team'], right_on=['date', 'team']).rename(columns={'goals_for':'home_goals_for',
                                                                                                           	'goals_against':'home_goals_against',
                                                                                                            'xG_for':'home_xG_for',	
                                                                                                            'xG_against':'home_xG_against'	,
                                                                                                            'is_home':'home_is_home',	
                                                                                                            'points gained':'home_points_gained',	
                                                                                                            'rest_days':'home_rest_days',	
                                                                                                            'total_goals':'home_total_goals',	
                                                                                                            'total_xg':'home_total_xg',	
                                                                                                            'total_goals_against':'home_total_goals_against',	
                                                                                                            'total_xg_against':'home_total_xg_against',	
                                                                                                            'last_5':'home_last_5',	
                                                                                                            'match_played':'home_match_played',	
                                                                                                            'PPG':'home_PPG',
                                                                                                            'avg_shots_for' : 'avg_home_shots',
                                                                                                            'avg_shots_target_for' : 'avg_home_shots_target',
                                                                                                            'avg_shots_for' : 'avg_home_shots_against',
                                                                                                            'avg_shots_target_against' : 'avg_home_target_shots_against',
                                                                                                            'avg_corners_for' : 'avg_home_corners_for',
                                                                                                            'avg_corners_against' : 'avg_home_corners_against'})
 
 
df = pd.merge(df, df_long, left_on=['date', 'away_team'], right_on=['date', 'team']).rename(columns={'goals_for':'away_goals_for',
                                                                                                           	'goals_against':'away_goals_against',
                                                                                                            'xG_for':'away_xG_for',	
                                                                                                            'xG_against':'away_xG_against'	,
                                                                                                            'is_home':'away_is_home',	
                                                                                                            'points gained':'away_points_gained',	
                                                                                                            'rest_days':'away_rest_days',	
                                                                                                            'total_goals':'away_total_goals',	
                                                                                                            'total_xg':'away_total_xg',	
                                                                                                            'total_goals_against':'away_total_goals_against',	
                                                                                                            'total_xg_against':'away_total_xg_against',	
                                                                                                            'last_5':'away_last_5',	
                                                                                                            'match_played':'away_match_played',	
                                                                                                            'PPG':'away_PPG',
                                                                                                            'avg_shots_for' : 'avg_away_shots',
                                                                                                            'avg_shots_target_for' : 'avg_away_shots_target',
                                                                                                            'avg_shots_for' : 'avg_away_shots_against',
                                                                                                            'avg_shots_target_against' : 'avg_away_target_shots_against',
                                                                                                            'avg_corners_for' : 'avg_away_corners_for',
                                                                                                            'avg_corners_against' : 'avg_away_corners_against'})


df = df.dropna()

df = df.reset_index(drop=True)


df.to_csv('data/final_data.csv', sep=';', index=False)

print('Data frame with shape {} correctly saved as CSV'.format(df.shape))


Data frame with shape (7636, 80) correctly saved as CSV


In [21]:
df.head()

,season_x,date,game,home_team,away_team,home_goals,away_goals,home_xg,away_xg,Date,...,away_total_xg_against,away_last_5,away_match_played,away_PPG,avg_away_shots_against,avg_away_shots_target,avg_shots_against_y,avg_away_target_shots_against,avg_away_corners_for,avg_away_corners_against
0,2122,2021-09-21,2021-09-21 Athletic Club-Rayo Vallecano,Ath Bilbao,Vallecano,1.0,2.0,0.956027,1.041210,2021-09-21,...,8.793662,7.0,5,1.4,13.0,4.2,12.6,4.8,3.8,4.0
1,2122,2021-09-21,2021-09-21 Getafe-Atletico Madrid,Getafe,Ath Madrid,1.0,2.0,0.330581,1.479060,2021-09-21,...,3.690765,11.0,5,2.2,14.4,3.6,6.2,1.8,6.2,2.8
2,2122,2021-09-21,2021-09-21 Levante-Celta Vigo,Levante,Celta,0.0,2.0,1.407020,0.739683,2021-09-21,...,9.950860,1.0,5,0.2,10.4,2.0,11.2,5.4,3.8,3.6
3,2122,2021-09-22,2021-09-22 Real Madrid-Mallorca,Real Madrid,Mallorca,6.0,1.0,2.374600,1.191650,2021-09-22,...,5.521310,8.0,5,1.6,9.4,2.4,11.4,3.2,4.0,2.8
4,2122,2021-09-23,2021-09-23 Granada-Real Sociedad,Granada,Sociedad,2.0,3.0,1.250190,1.581530,2021-09-23,...,6.122238,10.0,5,2.0,11.6,4.2,9.0,2.8,4.6,3.8


## 2. The Predictive Model

Having all the data is crucial, now let's get to the funny stuff: it's time to find a nice model.ù

After all the preprocessing and the feature engineering it's time to perform a little validation to find the best model and its best hyperparameters.